In [ ]:
import numpy as np
import pandas as pd
import pickle as pkl
from pathlib import Path

from topostats.filters import Filters
from topostats.grains import Grains
from topostats.grainstats import GrainStats
from topostats.io import find_files, read_yaml, write_yaml, LoadScans
from topostats.logs.logs import setup_logger, LOGGER_NAME
from topostats.plottingfuncs import Images
from topostats.tracing.dnatracing import dnaTrace, crop_array, prep_arrays  # , traceStats
from topostats.utils import update_config

In [ ]:
BASE_DIR = Path().cwd()
# Alternatively if you know where your files area comment the above line and uncomment the below adjust it for your use.
# BASE_DIR = Path("/path/to/where/my/files/are")
# Adjust the file extension approriately.
FILE_EXT = ".spm"
# Search for *.spm files one directory level up from the current notebooks
image_files = find_files(base_dir=BASE_DIR.parent / "tests", file_ext=FILE_EXT)
config = read_yaml(BASE_DIR.parent / "topostats" / "default_config.yaml")
loading_config = config["loading"]
filter_config = config["filter"]
filter_config.pop("run")
grain_config = config["grains"]
grain_config.pop("run")
grainstats_config = config["grainstats"]
grainstats_config.pop("run")
dnatracing_config = config["dnatracing"]
dnatracing_config.pop("run")
plotting_config = config["plotting"]
plotting_config.pop("run")
plotting_config["image_set"] = "all"

In [ ]:
all_scan_data = LoadScans(image_files, **config["loading"])
all_scan_data.get_data()
filtered_image = Filters(
    image=all_scan_data.img_dict["minicircle"]["image"],
    filename=all_scan_data.img_dict["minicircle"]["img_path"],
    pixel_to_nm_scaling=all_scan_data.img_dict["minicircle"]["px_2_nm"],
    **filter_config,
)
filtered_image.filter_image()
fig, ax = Images(
    data=filtered_image.images["gaussian_filtered"],
    filename=filtered_image.filename,
    output_dir="img/",
    save=True,
    **plotting_config,
).plot_and_save()
fig

In [ ]:
grains = Grains(
    image=filtered_image.images["zero_average_background"],
    filename=filtered_image.filename,
    pixel_to_nm_scaling=filtered_image.pixel_to_nm_scaling,
    **grain_config,
)
grains.find_grains()
print(
    f"Available NumPy arrays to plot in grains.directions['above'] dictionary :\n\n{grains.directions['above'].keys()}"
)

In [ ]:
plotting_config["colorbar"] = False
fig, ax = Images(
    data=grains.directions["above"]["coloured_regions"],
    filename=filtered_image.filename,
    output_dir="img/",
    save=True,
    **plotting_config,
).plot_and_save()
fig

In [ ]:
import skimage.measure as skimage_measure
import pickle as pkl
from pathlib import Path

region_properties = skimage_measure.regionprops(grains.directions["above"]["labelled_regions_02"])
# def crop_array(array: np.ndarray, bounding_box) -> np.ndarray:
#     """Crop an array.

#         Parameters
#         ----------
#         array: np.ndarray
#             2D Numpy array to be cropped.
#         bounding_box: Tuple
#             Tuple of co-ordinates to crop, should be of form (max_x, min_x, max_y, min_y) as returned by
#         _get_bounding_box().

#         Returns
#         -------
#         np.ndarray()
#             Cropped array
#         """

#     return array[bounding_box[0] - 30:bounding_box[2] - 30, bounding_box[1] + 30:bounding_box[3] + 30]
n_grain = 0
grain_arrays = []
output_dir = BASE_DIR.parent / "tmp" / "individual_grains"
cropped_array, cropped_mask = prep_arrays(
    filtered_image.images["gaussian_filtered"], grains.directions["above"]["labelled_regions_02"], pad_width=30
)
for grain in region_properties:
    # cropped_array = crop_array(filtered_image.images["gaussian_filtered"], grain.bbox)
    # cropped_mask = crop_array(grains.directions["above"]["labelled_regions_02"], grain.bbox)
    grain_arrays = {"original": cropped_array[n_grain], "mask": cropped_mask[n_grain]}
    print(f"\n\nGrain            : {n_grain}")
    print(f"Bounding Box     : {grain.bbox}")
    print(f"Original array   : {cropped_array[n_grain].shape}")
    print(f"Mask array       : {cropped_mask[n_grain].shape}")
    # print(cropped_array)
    outfile = output_dir / f"grain_{n_grain}.pkl"
    with outfile.open("wb") as output:
        pkl.dump(grain_arrays, output)
    n_grain += 1

In [ ]:
## Load and plot single grain
from typing import Tuple

dnatracing_config.pop("pad_width")


def load_and_plot(grain_n: int = 8, pad: int = 10) -> Tuple:
    single_grain_pkl = BASE_DIR.parent / "tmp" / "individual_grains" / f"grain_{grain_n}.pkl"
    with single_grain_pkl.open("rb") as input:
        single_grain = pkl.load(input)
    image = single_grain["original"]
    mask = single_grain["mask"]
    print(f"Unique labells in bounding box : {np.unique(mask)}")
    fig, ax = Images(
        data=image,
        filename=f"grain{grain_n}",
        output_dir="../tmp/individual_grains/",
        save=True,
        **plotting_config,
    ).plot_and_save()
    dna_traces = dnaTrace(
        image=np.pad(image, pad),
        grain=np.pad(mask, pad),
        filename=grains.filename.stem,
        pixel_to_nm_scaling=grains.pixel_to_nm_scaling,
        **dnatracing_config,
    )
    dna_traces.trace_dna()
    return image, mask, fig


image_3, mask_3, fig_3 = load_and_plot(grain_n=3)
fig_3

In [ ]:
image_9, mask_9, fig_9 = load_and_plot(grain_n=9)
fig_9

In [ ]:
pad = 10
# Linear molecule
np.save(BASE_DIR.parent / "tests" / "resources" / "dnatracing_mask_linear.npy", mask_3)
np.save(BASE_DIR.parent / "tests" / "resources" / "dnatracing_image_linear.npy", image_3)
pad = 20
np.save(BASE_DIR.parent / "tests" / "resources" / "dnatracing_mask_circular.npy", mask_9)
np.save(BASE_DIR.parent / "tests" / "resources" / "dnatracing_image_circular.npy", image_9)

In [ ]:
dnatracing_config
grains.pixel_to_nm_scaling

In [ ]:
# from topostats.tracing.dnatracing import tracedna
def tracedna(image: np.ndarray, grains: np.ndarray, cores: int = 1) -> pd.DataFrame:
    """Convenience function for processing multiple grains individually.

    Parameters
    ----------
    image : np.ndarray
        Full image as Numpy Array.
    grains : np.ndarray
        Full image as Grains that are labelled.
    cores : int
        Number of cores to process with.

    Returns
    -------
    pd.DataFrame
        Statistics from skeletonising and tracing the grains in the image.

    """
    # Check both arrays are the same shape
    assert image.shape == grains.shape

    # Get bounding boxes for each grain
    region_properties = skimage_measure.regionprops(grains)
    # Subset image and grains then zip them up
    cropped_image = [crop_array(image, grain.bbox) for grain in region_properties]
    cropped_mask = [crop_array(grains, grain.bbox) for grain in region_properties]
    # Flip every labelled region to be 1 instead of its label
    cropped_mask = [np.where(grain == 0, 0, 1) for grain in cropped_mask]
    print(cropped_image)
    print(cropped_mask)


tracedna(image=filtered_image.images["gaussian_filtered"], grains=grains.directions["above"]["labelled_regions_02"])